# Unified-IO 2 多模态统一模型教程

本教程覆盖 Unified-IO 2 的核心思想、结构与最小可运行示例，重点演示：

1. **任意模态输入**：文本 / 图像 / 音频 / 视频
2. **统一表示空间**：共享编码器与模态嵌入
3. **统一任务接口**：分类、检索、生成
4. **最小可运行代码**：随机数据验证形状与流程

---

## 环境设置

In [ ]:
import sys
from pathlib import Path

# 添加 src 目录到路径
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import torch
import torch.nn.functional as F

from unified_io import (
    UnifiedIOConfig,
    UnifiedIO,
    MultimodalBatch,
    TaskType,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## 1. Unified-IO 2 的核心理念

Unified-IO 2 将多模态学习抽象为统一的 **token 流**，用同一套编码器与解码器处理不同模态：

```
Text/Image/Audio/Video  →  Patch/Token Embedding  →  Unified Encoder  →  Unified Representation
                                                      ↘ Task Heads (cls/retrieval)
                                                       ↘ Unified Decoder (generation)
```

关键点：
- 使用 **模态嵌入** 标识 token 来源
- 所有模态拼接进入同一 Transformer
- 通过统一任务入口输出分类 / 检索 / 生成结果


## 2. 配置与模型初始化

In [ ]:
config = UnifiedIOConfig(
    hidden_size=384,
    num_layers=4,
    num_heads=6,
    max_text_length=32,
    image_size=224,
    image_patch_size=16,
    audio_patch_size=16,
    video_frames=4,
    num_labels=5,
)

model = UnifiedIO(config).to(device)
print(model.__class__.__name__)


## 3. 构造多模态输入

我们使用随机张量模拟不同模态输入，重点验证 token 拼接与形状变化。


In [ ]:
batch_size = 2
text_ids = torch.randint(0, config.text_vocab_size, (batch_size, 12)).to(device)
text_mask = torch.ones(batch_size, 12).to(device)

images = torch.randn(batch_size, 3, config.image_size, config.image_size).to(device)
audio = torch.randn(batch_size, config.max_audio_length).to(device)
video = torch.randn(batch_size, 3, config.video_frames, config.image_size, config.image_size).to(device)

batch = MultimodalBatch(
    text_input_ids=text_ids,
    text_attention_mask=text_mask,
    images=images,
    audio=audio,
    video=video,
)
batch


## 4. 编码器输出与 token span

In [ ]:
encoder_out = model.encoder(batch)
hidden = encoder_out['hidden_states']
spans = encoder_out['spans']

print('Hidden shape:', hidden.shape)
print('Spans:', spans)


## 5. 统一任务接口

### 5.1 分类任务
使用 [CLS] 或平均池化向量输出分类 logits。


In [ ]:
labels = torch.randint(0, config.num_labels, (batch_size,)).to(device)
batch.labels = labels

outputs = model(batch, task=TaskType.CLASSIFICATION)
print('Logits shape:', outputs['logits'].shape)
print('Loss:', outputs['loss'].item())


### 5.2 检索任务
输出归一化的统一 embedding，可用于跨模态检索。


In [ ]:
retrieval_out = model(batch, task=TaskType.RETRIEVAL)
embeddings = retrieval_out['embeddings']
print('Embedding shape:', embeddings.shape)
print('Embedding norm:', embeddings.norm(dim=-1))


### 5.3 生成任务
使用统一解码器进行自回归生成。这里用文本前缀演示。


In [ ]:
decoder_labels = torch.randint(0, config.text_vocab_size, (batch_size, 12)).to(device)
batch.labels = decoder_labels

gen_out = model(batch, task=TaskType.GENERATION)
print('Generation logits:', gen_out['logits'].shape)


## 6. 简单的生成示例

In [ ]:
batch.labels = None
prefix = torch.randint(0, config.text_vocab_size, (batch_size, 4)).to(device)
batch.text_input_ids = prefix

generated = model.generate(batch, max_length=8)
print('Generated token ids:', generated)


## 7. 关键实现细节回顾

- **PatchEmbed2D/1D/3D** 负责将图像、音频、视频转换为 token 序列
- **ModalityEmbedding** 提供模态标识，让 Transformer 知道来源
- **UnifiedEncoder** 负责跨模态交互
- **UnifiedDecoder** 负责生成任务输出

后续可扩展：
- 真实数据加载与预处理
- 多任务训练与损失设计
- 更大规模参数与稀疏注意力
